# 🦵 RSNA Knee Multimodal AI - Phase 3: MRI DICOM & Volumetric Preprocessing Pipeline

This notebook demonstrates the end-to-end medical image preprocessing and PyTorch data pipeline for multi-sequence knee MRI studies.

### Pipeline Highlights:
1. **Intelligent Series Selector & Ranker**: Selects optimal series for `Sagittal`, `Coronal`, and `Axial` views prioritizing fluid-sensitive and fat-suppressed sequences.
2. **DICOM Reader & Metadata Parser**: Normalizes slope/intercept, handles photometric interpretation, and performs anatomical slice sorting.
3. **Volumetric Standardization**: Dynamic percentile-based windowing $[0, 1]$, depth resampling ($D=16$), and spatial resizing $(224 \times 224)$.
4. **3D Consistent Augmentations**: Anatomically consistent spatial affine transforms and intensity jitter across volume depth.
5. **Multi-View PyTorch Dataset & DataLoaders**: Leak-free 5-fold cross-validation batch generator.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is accessible
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src.mri_preprocessor import (
    DICOMReader, 
    IntensityNormalizer, 
    SeriesSelector, 
    VolumeResampler, 
    SyntheticDICOMGenerator, 
    TARGET_COLS, 
    PLANES
)
from src.dataset import (
    MRIVolumeAugmentations, 
    KneeMRIDataset, 
    knee_collate_fn, 
    create_dataloaders
)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

## 1. 🔍 Series Selector & Multi-Sequence Ranking
In knee MRI, different sequences capture distinct anatomical features. Fluid-sensitive (T2/PD) and fat-suppressed sequences are optimal for identifying ligament tears, cartilage erosion, and bone contusions.

In [ ]:
series_df = pd.read_csv('../data/train_series.csv')
selector = SeriesSelector(series_df)

sample_study = series_df['StudyInstanceUID'].iloc[0]
print(f"Selected Study: {sample_study}\n")

study_series = selector.get_study_series_info(sample_study)
print("--- Available Series in Study ---")
print(study_series[['SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression', 'Priority_Score']])

selected_planes = selector.select_study_series(sample_study)
print("\n--- Optimal Series Selected per Plane ---")
for plane, s_uid in selected_planes.items():
    print(f"{plane:10s} -> {s_uid}")

## 2. 🩻 Synthetic DICOM Generation & Volume Reconstruction
Let's generate synthetic DICOM series for all 3 orthogonal planes (`Sagittal`, `Coronal`, `Axial`) and visualize the reconstructed 3D volumes.

In [ ]:
temp_dir = Path('../outputs/demo_dicoms')
study_id = "demo_study_101"

planes_data = {}
for p in PLANES:
    s_uid = f"series_{p.lower()}_101"
    series_dir = SyntheticDICOMGenerator.save_mock_dicom_series(
        output_dir=temp_dir,
        study_uid=study_id,
        series_uid=s_uid,
        depth=16,
        plane=p
    )
    vol, metas = DICOMReader.read_series(series_dir)
    planes_data[p] = VolumeResampler.preprocess_volume(vol, target_depth=16, target_spatial=(224, 224))

# Visualize central slices from each plane
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, p in enumerate(PLANES):
    mid_slice = planes_data[p][8]  # Center slice index
    axes[idx].imshow(mid_slice, cmap='bone')
    axes[idx].set_title(f"{p} View (Center Slice 8/16)", fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3. 🔄 3D-Consistent Volumetric Data Augmentations
Medical MRI volumes must preserve geometric consistency across slices during transformations.

In [ ]:
aug_pipeline = MRIVolumeAugmentations(
    p_flip=0.5,
    p_rot=0.7,
    max_rot_deg=12.0,
    p_scale_shift=0.6,
    p_gamma=0.6,
    p_noise=0.5,
    p_cutout=0.5,
    is_train=True
)

orig_vol = planes_data['Sagittal']
aug_vol = aug_pipeline(orig_vol, plane='Sagittal')

# Display multi-slice comparison
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
slices_to_view = [2, 6, 10, 14]

for i, s_idx in enumerate(slices_to_view):
    axes[0, i].imshow(orig_vol[s_idx], cmap='bone')
    axes[0, i].set_title(f"Original (Slice {s_idx})", fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(aug_vol[s_idx], cmap='bone')
    axes[1, i].set_title(f"Augmented (Slice {s_idx})", fontsize=10)
    axes[1, i].axis('off')

plt.suptitle("Original vs 3D-Augmented Volume Slices", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. ⚡ PyTorch DataLoader Batch Pipeline Verification
Let's instantiate the multi-view `KneeMRIDataset` and cross-validation `DataLoader`.

In [ ]:
train_loader, val_loader = create_dataloaders(
    df_path='../data/train_folds.csv',
    series_path='../data/train_series.csv',
    val_fold=0,
    batch_size=4,
    num_workers=0,
    target_depth=16,
    target_spatial=(224, 224),
    mode='pseudo_labeled'
)

batch = next(iter(train_loader))

print("=== Mini-Batch Tensor Specifications ===")
print(f"• Stacked Tri-Plane Tensor: {batch['image'].shape}  [B, 3 Planes, Depth, Height, Width]")
print(f"• Sagittal Plane Tensor:    {batch['sagittal'].shape}")
print(f"• Coronal Plane Tensor:     {batch['coronal'].shape}")
print(f"• Axial Plane Tensor:       {batch['axial'].shape}")
print(f"• Multi-Label Targets:      {batch['targets'].shape}  [B, 12 Pathologies]")
print(f"• Target Data Type:         {batch['targets'].dtype}")
print(f"• Has Gold Ground Truth:    {batch['has_gold'].tolist()}")

print("\n=== Sample Target Probabilities for Study 0 ===")
sample_probs = batch['targets'][0].numpy()
for path, prob in zip(TARGET_COLS, sample_probs):
    print(f"{path:20s}: {prob:.3f}")